In [17]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN,Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [18]:
## ---Data---

sentences = [
    "I love this course",
    "This is amazing",
    "I hate this topic",
    "This is boring"
]

labels = np.array([1,1,0,0]) # 1 = positive, 0 = negative

In [19]:
# Tokenzation
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)

sequences = tokenizer.texts_to_sequences(sentences)
padded = pad_sequences(sequences, padding = "post")

vocab_size = len(tokenizer.word_index) + 1
max_len = padded.shape[1]

In [20]:
# ---SIMPLE RNN WITHOUT EMBEDDING
model_no_embedding = Sequential()
model_no_embedding.add(SimpleRNN(10, input_shape= (max_len, 1)))
model_no_embedding.add(Dense(1, activation = "sigmoid"))

model_no_embedding.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

In [21]:
# RNN expects 3D input -> reshape
padded_reshaped = padded.reshape((padded.shape[0], padded.shape[1], 1))

In [22]:
# --- Training---
print("Training RNN WITHOUT Embedding")
model_no_embedding.fit(
    padded_reshaped,
    labels,
    epochs = 50,  # increased epochs for better learning on small dataset
    verbose = 0
)

Training RNN WITHOUT Embedding


In [23]:
# --- Evaluate on Training data---
loss, accuracy = model_no_embedding.evaluate(padded_reshaped, labels, verbose=0)
print(f"\nTraining Accuracy: {accuracy*100:.2f}%")


Training Accuracy: 50.00%


In [24]:
# --- Test with new input sentences---
test_sentences = [
    "I love this topic",
    "This is terrible",
    "I hate this course"
]

In [25]:
# Convert to sequences and pad
test_seq = tokenizer.texts_to_sequences(test_sentences)
test_padded = pad_sequences(test_seq, maxlen = max_len, padding = "post")
test_padded_reshaped = test_padded.reshape((test_padded.shape[0], test_padded.shape[1], 1))

In [26]:
# Predict
predictions = model_no_embedding.predict(test_padded_reshaped)
for sentence, pred in zip(test_sentences, predictions):
    label = 1 if pred >= 0.5 else 0
    print(f"Sentence: '{sentence}' -> Predicited: {label} (Probability: {pred[0]:.2f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 236ms/step
Sentence: 'I love this topic' -> Predicited: 0 (Probability: 0.19)
Sentence: 'This is terrible' -> Predicited: 1 (Probability: 0.67)
Sentence: 'I hate this course' -> Predicited: 0 (Probability: 0.24)
